# Capstone: Designing Castalia Scholar

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/32_project_architecture.ipynb)

## Module 7 — Capstone Project (Notebook 1 of 6)

Welcome to the capstone project! Over the next six notebooks, you will build **Castalia Scholar** — a multi-agent research assistant that takes a research question and produces a **cited, structured research report**.

This is *not* a toy demo. Castalia Scholar integrates every pattern from the course:
- **ReAct** (nb 02) for tool-using agents
- **Plan-and-Execute** (nb 06) for the planner
- **Reflection** (nb 07) for self-improvement
- **Multi-agent orchestration** (nb 08) for coordination
- **Agentic RAG** (rag module) for retrieval
- **Evaluation** (nb 26) for quality assessment

### What You'll Build in This Notebook

1. **System architecture** — the full design of Castalia Scholar
2. **Agent role definitions** — what each agent does, needs, and produces
3. **Message format** — standardized inter-agent communication
4. **Shared state schema** — the blackboard that all agents read/write
5. **Component interfaces** — abstract base classes for every agent
6. **Skeleton implementation** — all classes with stub methods
7. **Workflow design** — execution order, decision points, feedback loops
8. **Testing strategy** — how to verify each component

No heavy LLM inference here — this notebook is about **design and architecture**.

---

> **Prerequisites**: Complete Modules 1–6 and the RAG module.
> **Runtime**: Google Colab with T4 GPU (light usage — mostly design code).
> **Time**: ~30–40 minutes.

## Part 0: Environment Setup

We load the standard environment. This notebook is mostly design and skeleton code, so LLM usage is minimal.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Project Overview — What Is Castalia Scholar?

### 1.1 — The Problem

A researcher asks a complex question like:
> *"What are the main approaches to making AI systems more energy-efficient?"*

Answering this well requires:
1. **Planning** — decompose the question into sub-questions
2. **Retrieval** — find relevant documents for each sub-question
3. **Analysis** — synthesize findings, detect contradictions, identify gaps
4. **Writing** — produce a structured report with proper citations
5. **Review** — evaluate quality, request revisions if needed

No single LLM call can do all this reliably. We need **specialized agents** working together.

### 1.2 — The Solution: Multi-Agent Architecture

Castalia Scholar uses **five specialized agents** coordinated by an **orchestrator**:

| Agent | Responsibility | Key Patterns Used |
|-------|---------------|-------------------|
| **Planner** | Decompose question into sub-questions | Plan-and-Execute (nb 06) |
| **Retriever** | Find relevant documents with citations | Agentic RAG (rag module) |
| **Analyzer** | Synthesize, detect contradictions, find gaps | Reflection (nb 07), Critique |
| **Writer** | Produce structured report with citations | Plan-and-Execute, Iterative refinement |
| **Reviewer** | Evaluate quality, request revisions | Reflection (nb 07), Evaluation (nb 26) |
| **Orchestrator** | Coordinate all agents, manage workflow | Multi-agent patterns (nb 08) |

### 1.3 — The Capstone Notebooks

| Notebook | Focus |
|----------|-------|
| **32** (this one) | Architecture and interfaces |
| **33** | Retrieval Agent — agentic RAG with citations |
| **34** | Analysis Agent — synthesis and contradiction detection |
| **35** | Writing Agent — structured report generation |
| **36** | Review Agent — multi-dimensional evaluation |
| **37** | Full System — orchestration and end-to-end runs |

## 2. System Architecture

### 2.1 — Architecture Diagram

```
                          ┌─────────────────┐
                          │   User Query     │
                          │  "What are the   │
                          │  approaches to   │
                          │  energy-efficient │
                          │  AI?"            │
                          └────────┬─────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │       ORCHESTRATOR            │
                    │  Manages workflow, state,     │
                    │  feedback loops               │
                    └──┬───┬───┬───┬───┬───────────┘
                       │   │   │   │   │
          ┌────────────┘   │   │   │   └────────────┐
          ▼                ▼   │   ▼                ▼
    ┌──────────┐   ┌──────────┐│ ┌──────────┐ ┌──────────┐
    │ PLANNER  │   │RETRIEVER ││ │ WRITER   │ │ REVIEWER │
    │          │   │          ││ │          │ │          │
    │ Decompose│   │ Search & ││ │ Produce  │ │ Evaluate │
    │ question │   │ cite docs││ │ report   │ │ & score  │
    └──────────┘   └──────────┘│ └──────────┘ └──────────┘
                               ▼
                         ┌──────────┐
                         │ ANALYZER │
                         │          │
                         │Synthesize│
                         │& detect  │
                         │conflicts │
                         └──────────┘

    Data Flow:
    ═════════
    User Query ──► Planner ──► Sub-questions
    Sub-questions ──► Retriever ──► Retrieved Documents + Citations
    Retrieved Docs ──► Analyzer ──► Analysis (findings, contradictions, gaps)
    Analysis ──► Writer ──► Draft Report
    Draft Report ──► Reviewer ──► Feedback + Scores
    [If revision needed] Feedback ──► Writer ──► Revised Report ──► Reviewer
    [If acceptable] ──► Final Report ──► User
```

### 2.2 — Design Principles

1. **Separation of concerns** — each agent has one job
2. **Standardized communication** — all agents speak the same message format
3. **Shared state (blackboard)** — agents read/write to a common state object
4. **Feedback loops** — the reviewer can send reports back for revision
5. **Bounded iteration** — max 2 revision cycles to prevent infinite loops
6. **Testability** — each agent can be tested independently

## 3. Agent Role Definitions

Each agent has a clear contract: what it receives, what it produces, and what tools it needs.

In [ ]:
# ─── Agent Role Definitions ───

AGENT_ROLES = {
    "planner": {
        "responsibility": "Decompose a research question into 3-5 focused sub-questions",
        "input": {
            "research_question": "str — the user's original question"
        },
        "output": {
            "sub_questions": "List[str] — focused sub-questions that together cover the topic",
            "search_queries": "List[str] — optimized search queries for each sub-question"
        },
        "tools": ["LLM"],
        "patterns": ["Plan-and-Execute (nb 06)"]
    },
    "retriever": {
        "responsibility": "Find relevant documents for each sub-question with citation tracking",
        "input": {
            "query": "str — a search query or sub-question",
            "max_results": "int — maximum documents to return (default 5)"
        },
        "output": {
            "results": "List[RetrievedDoc] — documents with content, source, relevance score",
            "citations": "List[Citation] — citation objects for attribution"
        },
        "tools": ["LLM", "FAISS index", "Embedding model"],
        "patterns": ["Agentic RAG (rag module)", "Self-RAG", "CRAG"]
    },
    "analyzer": {
        "responsibility": "Synthesize findings, detect contradictions, identify gaps",
        "input": {
            "documents": "List[RetrievedDoc] — retrieved documents to analyze",
            "research_question": "str — the original question for context"
        },
        "output": {
            "findings": "List[Finding] — key findings grouped by theme",
            "contradictions": "List[Contradiction] — conflicting claims between sources",
            "gaps": "List[str] — aspects of the question not covered",
            "evidence_quality": "Dict[str, float] — source quality ratings"
        },
        "tools": ["LLM"],
        "patterns": ["Reflection (nb 07)", "Critique patterns"]
    },
    "writer": {
        "responsibility": "Produce a structured research report with inline citations",
        "input": {
            "analysis": "AnalysisResult — synthesized findings and themes",
            "research_question": "str — for context and introduction"
        },
        "output": {
            "report": "Report — structured report with sections and citations",
            "references": "List[Citation] — bibliography"
        },
        "tools": ["LLM"],
        "patterns": ["Plan-and-Execute (nb 06)", "Iterative refinement (nb 09)"]
    },
    "reviewer": {
        "responsibility": "Evaluate report quality on multiple dimensions, decide if revision needed",
        "input": {
            "report": "Report — the draft report to evaluate",
            "rubric": "Dict — scoring rubric with dimensions and criteria"
        },
        "output": {
            "scores": "Dict[str, int] — score (1-5) for each dimension",
            "feedback": "List[str] — specific, actionable feedback items",
            "needs_revision": "bool — whether the report needs another pass"
        },
        "tools": ["LLM"],
        "patterns": ["Reflection (nb 07)", "Evaluation (nb 26)"]
    }
}

# Display the role definitions
for agent_name, role in AGENT_ROLES.items():
    print(f"\n{'='*60}")
    print(f"  {agent_name.upper()}")
    print(f"{'='*60}")
    print(f"  Responsibility: {role['responsibility']}")
    print(f"  Inputs:  {list(role['input'].keys())}")
    print(f"  Outputs: {list(role['output'].keys())}")
    print(f"  Tools:   {role['tools']}")
    print(f"  Patterns: {role['patterns']}")

## 4. Inter-Agent Message Format

All agents communicate through a standardized message format. This makes the system modular — you can swap out any agent without changing the others.

In [ ]:
# ─── Standardized Message Format ───

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
import time as _time
import uuid

@dataclass
class AgentMessage:
    """Standard message format for inter-agent communication."""
    from_agent: str          # sender agent name
    to_agent: str            # recipient agent name
    msg_type: str            # message type: "request", "response", "feedback", "error"
    content: Dict[str, Any]  # payload — varies by message type
    msg_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    timestamp: float = field(default_factory=_time.time)
    in_reply_to: Optional[str] = None  # msg_id of the message this replies to

    def to_dict(self) -> dict:
        return {
            "from": self.from_agent,
            "to": self.to_agent,
            "type": self.msg_type,
            "content": self.content,
            "msg_id": self.msg_id,
            "timestamp": self.timestamp,
            "in_reply_to": self.in_reply_to
        }

    def __repr__(self):
        return (f"Message({self.from_agent} -> {self.to_agent}, "
                f"type={self.msg_type}, id={self.msg_id})")

# Example messages
plan_request = AgentMessage(
    from_agent="orchestrator",
    to_agent="planner",
    msg_type="request",
    content={"research_question": "What are the main approaches to energy-efficient AI?"}
)

plan_response = AgentMessage(
    from_agent="planner",
    to_agent="orchestrator",
    msg_type="response",
    content={
        "sub_questions": [
            "What hardware approaches reduce AI energy consumption?",
            "How do model compression techniques improve efficiency?",
            "What algorithmic innovations reduce computational cost?"
        ],
        "search_queries": [
            "energy efficient AI hardware accelerators",
            "model pruning quantization distillation",
            "efficient training algorithms green AI"
        ]
    },
    in_reply_to=plan_request.msg_id
)

print("Example: Planning Request")
print(json.dumps(plan_request.to_dict(), indent=2, default=str))
print("\nExample: Planning Response")
print(json.dumps(plan_response.to_dict(), indent=2, default=str))

## 5. Shared State — The Blackboard

All agents read from and write to a shared state object. This is the **blackboard pattern** — a common data structure that coordinates agent collaboration without direct agent-to-agent messaging.

In [ ]:
# ─── Shared State (Blackboard) ───

@dataclass
class SharedState:
    """
    The blackboard — shared state that all agents read/write.
    Each field represents one stage of the research pipeline.
    """
    # Input
    research_question: str = ""

    # Planner output
    sub_questions: List[str] = field(default_factory=list)
    search_queries: List[str] = field(default_factory=list)

    # Retriever output
    retrieved_docs: List[Dict[str, Any]] = field(default_factory=list)
    citations: List[Dict[str, str]] = field(default_factory=list)

    # Analyzer output
    findings: List[Dict[str, Any]] = field(default_factory=list)
    contradictions: List[Dict[str, Any]] = field(default_factory=list)
    gaps: List[str] = field(default_factory=list)

    # Writer output
    draft_report: str = ""
    report_sections: List[Dict[str, str]] = field(default_factory=list)
    references: List[Dict[str, str]] = field(default_factory=list)

    # Reviewer output
    review_scores: Dict[str, int] = field(default_factory=dict)
    review_feedback: List[str] = field(default_factory=list)
    needs_revision: bool = False

    # Final output
    final_report: str = ""
    revision_count: int = 0
    max_revisions: int = 2

    # Metadata
    message_log: List[Dict] = field(default_factory=list)
    agent_call_count: Dict[str, int] = field(default_factory=lambda: defaultdict(int))
    total_tokens: int = 0
    start_time: float = field(default_factory=_time.time)

    def log_message(self, msg: AgentMessage):
        """Record a message in the log."""
        self.message_log.append(msg.to_dict())
        self.agent_call_count[msg.from_agent] += 1

    def elapsed_time(self) -> float:
        return _time.time() - self.start_time

    def summary(self) -> str:
        """Print a summary of current state."""
        lines = [
            f"Research Question: {self.research_question[:80]}...",
            f"Sub-questions: {len(self.sub_questions)}",
            f"Retrieved docs: {len(self.retrieved_docs)}",
            f"Findings: {len(self.findings)}",
            f"Contradictions: {len(self.contradictions)}",
            f"Gaps: {len(self.gaps)}",
            f"Draft report: {'Yes' if self.draft_report else 'No'} ({len(self.draft_report)} chars)",
            f"Review scores: {self.review_scores or 'Not yet reviewed'}",
            f"Needs revision: {self.needs_revision}",
            f"Revision count: {self.revision_count}/{self.max_revisions}",
            f"Messages logged: {len(self.message_log)}",
            f"Agent calls: {dict(self.agent_call_count)}",
        ]
        return "\n".join(lines)


# Create a fresh shared state
state = SharedState(research_question="What are the main approaches to making AI systems more energy-efficient?")
print("Shared State Summary (initial):")
print(state.summary())

## 6. Component Interfaces — Abstract Base Classes

We define **abstract base classes (ABCs)** for each agent type. This ensures every agent implementation follows the same contract.

In [ ]:
# ─── Abstract Base Classes ───

from abc import ABC, abstractmethod

class BaseAgent(ABC):
    """Base class for all Castalia Scholar agents."""

    def __init__(self, name: str):
        self.name = name
        self.call_count = 0

    @abstractmethod
    def process(self, state: SharedState) -> AgentMessage:
        """
        Process the current shared state and return a message with results.
        Each agent reads what it needs from state and returns its output.
        """
        pass

    def _create_message(self, to: str, msg_type: str, content: dict,
                        in_reply_to: str = None) -> AgentMessage:
        """Helper to create a properly formatted message."""
        self.call_count += 1
        return AgentMessage(
            from_agent=self.name,
            to_agent=to,
            msg_type=msg_type,
            content=content,
            in_reply_to=in_reply_to
        )

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name}, calls={self.call_count})"


class PlannerInterface(BaseAgent):
    """Interface for the Planner agent."""

    @abstractmethod
    def decompose(self, question: str) -> Dict[str, Any]:
        """Decompose a research question into sub-questions and search queries."""
        pass


class RetrieverInterface(BaseAgent):
    """Interface for the Retriever agent."""

    @abstractmethod
    def retrieve(self, query: str, max_results: int = 5) -> List[Dict]:
        """Retrieve relevant documents for a query."""
        pass

    @abstractmethod
    def assess_relevance(self, doc: Dict, query: str) -> float:
        """Score a document's relevance to a query (0-1)."""
        pass


class AnalyzerInterface(BaseAgent):
    """Interface for the Analyzer agent."""

    @abstractmethod
    def synthesize(self, docs: List[Dict], question: str) -> List[Dict]:
        """Synthesize key findings from multiple documents."""
        pass

    @abstractmethod
    def detect_contradictions(self, docs: List[Dict]) -> List[Dict]:
        """Find contradicting claims across documents."""
        pass

    @abstractmethod
    def identify_gaps(self, docs: List[Dict], question: str) -> List[str]:
        """Identify aspects of the question not covered by documents."""
        pass


class WriterInterface(BaseAgent):
    """Interface for the Writer agent."""

    @abstractmethod
    def plan_structure(self, analysis: Dict) -> List[Dict]:
        """Create a report outline from the analysis."""
        pass

    @abstractmethod
    def write_section(self, section_plan: Dict, findings: List[Dict]) -> str:
        """Write one section of the report."""
        pass

    @abstractmethod
    def assemble_report(self, sections: List[str], question: str) -> str:
        """Assemble all sections into a complete report."""
        pass


class ReviewerInterface(BaseAgent):
    """Interface for the Reviewer agent."""

    @abstractmethod
    def review(self, report: str, rubric: Dict) -> Dict:
        """Evaluate a report on multiple dimensions."""
        pass

    @abstractmethod
    def needs_revision(self, scores: Dict, threshold: float = 3.5) -> bool:
        """Determine if the report needs revision based on scores."""
        pass


print("✓ All abstract base classes defined:")
for cls in [BaseAgent, PlannerInterface, RetrieverInterface,
            AnalyzerInterface, WriterInterface, ReviewerInterface]:
    methods = [m for m in dir(cls) if not m.startswith('_') and callable(getattr(cls, m, None))]
    abstract = list(cls.__abstractmethods__) if hasattr(cls, '__abstractmethods__') else []
    print(f"  {cls.__name__}: methods={methods}, abstract={abstract}")

## 7. Data Classes — Structured Data Types

These dataclasses define the structured data that flows between agents.

In [ ]:
# ─── Data Classes for Agent Communication ───

@dataclass
class RetrievedDoc:
    """A document retrieved by the Retriever agent."""
    doc_id: str
    title: str
    content: str
    source: str
    relevance_score: float = 0.0
    query: str = ""

    def to_dict(self) -> dict:
        return {
            "doc_id": self.doc_id, "title": self.title,
            "content": self.content, "source": self.source,
            "relevance_score": self.relevance_score, "query": self.query
        }

@dataclass
class Citation:
    """A citation linking a claim to its source."""
    citation_id: int
    doc_id: str
    title: str
    source: str
    excerpt: str = ""

    def format_inline(self) -> str:
        return f"[{self.citation_id}]"

    def format_reference(self) -> str:
        return f"[{self.citation_id}] {self.title}. Source: {self.source}"

@dataclass
class Finding:
    """A key finding extracted from analysis."""
    theme: str
    claim: str
    supporting_docs: List[str]  # doc_ids
    confidence: float = 0.0     # 0-1

@dataclass
class Contradiction:
    """A contradiction detected between sources."""
    claim_a: str
    source_a: str  # doc_id
    claim_b: str
    source_b: str  # doc_id
    description: str = ""

@dataclass
class AnalysisResult:
    """Complete analysis output."""
    findings: List[Finding] = field(default_factory=list)
    contradictions: List[Contradiction] = field(default_factory=list)
    gaps: List[str] = field(default_factory=list)
    evidence_quality: Dict[str, float] = field(default_factory=dict)

@dataclass
class ReportSection:
    """One section of the research report."""
    title: str
    content: str
    citations: List[int] = field(default_factory=list)  # citation_ids

@dataclass
class Report:
    """A complete research report."""
    title: str
    introduction: str = ""
    sections: List[ReportSection] = field(default_factory=list)
    conclusion: str = ""
    references: List[Citation] = field(default_factory=list)

    def full_text(self) -> str:
        parts = [f"# {self.title}\n", f"## Introduction\n{self.introduction}\n"]
        for s in self.sections:
            parts.append(f"## {s.title}\n{s.content}\n")
        parts.append(f"## Conclusion\n{self.conclusion}\n")
        parts.append("## References\n")
        for ref in self.references:
            parts.append(ref.format_reference())
        return "\n".join(parts)

@dataclass
class ReviewResult:
    """Review evaluation output."""
    scores: Dict[str, int] = field(default_factory=dict)
    feedback: List[str] = field(default_factory=list)
    needs_revision: bool = False
    overall_score: float = 0.0


# Demonstrate creating example data
example_doc = RetrievedDoc(
    doc_id="doc_001", title="Energy-Efficient Neural Architecture Search",
    content="NAS techniques can find architectures that are both accurate and energy-efficient...",
    source="Chen et al., 2023, ICML", relevance_score=0.92,
    query="energy efficient AI architectures"
)

example_citation = Citation(
    citation_id=1, doc_id="doc_001",
    title="Energy-Efficient Neural Architecture Search",
    source="Chen et al., 2023, ICML",
    excerpt="NAS techniques can find architectures that are both accurate and energy-efficient"
)

print("Data class examples:")
print(f"  Document: {example_doc.title} (relevance: {example_doc.relevance_score})")
print(f"  Citation: {example_citation.format_inline()} -> {example_citation.format_reference()}")
print(f"\n✓ All data classes defined: RetrievedDoc, Citation, Finding, Contradiction,")
print(f"  AnalysisResult, ReportSection, Report, ReviewResult")

## 8. Skeleton Implementations

Now we create concrete skeleton classes that implement the interfaces. Each method has the correct signature and a stub implementation. These will be fully implemented in notebooks 33–37.

In [ ]:
# ─── Skeleton: PlannerAgent ───

class PlannerAgent(PlannerInterface):
    """Decomposes a research question into sub-questions."""

    def __init__(self):
        super().__init__(name="planner")

    def decompose(self, question: str) -> Dict[str, Any]:
        """
        Use LLM to break the question into sub-questions.
        Returns {"sub_questions": [...], "search_queries": [...]}.
        """
        prompt = f"""Decompose this research question into 3-5 focused sub-questions.
For each sub-question, also provide an optimized search query.

Research question: {question}

Format your response as JSON:
{{"sub_questions": ["...", "..."], "search_queries": ["...", "..."]}}"""

        messages = [{"role": "user", "content": prompt}]
        response = generate(messages, max_new_tokens=400, temperature=0.3)

        try:
            result = json.loads(response)
        except json.JSONDecodeError:
            match = re.search(r'\{.*\}', response, re.DOTALL)
            result = json.loads(match.group()) if match else {
                "sub_questions": [question],
                "search_queries": [question]
            }
        return result

    def process(self, state: SharedState) -> AgentMessage:
        """Main entry point — decompose the research question."""
        result = self.decompose(state.research_question)
        state.sub_questions = result.get("sub_questions", [])
        state.search_queries = result.get("search_queries", [])
        return self._create_message(
            to="orchestrator", msg_type="response",
            content=result
        )

print("✓ PlannerAgent skeleton defined")
planner = PlannerAgent()
print(f"  {planner}")

In [ ]:
# ─── Skeleton: RetrievalAgent ───

class RetrievalAgent(RetrieverInterface):
    """Retrieves relevant documents using agentic RAG patterns."""

    def __init__(self, documents=None):
        super().__init__(name="retriever")
        self.documents = documents or []
        self.index = None  # FAISS index — built in nb 33

    def build_index(self, documents: List[Dict]):
        """Build FAISS index from documents. Implemented in nb 33."""
        self.documents = documents
        # Will use embed_texts() + faiss.IndexFlatIP in nb 33
        print(f"  [stub] Would build index from {len(documents)} documents")

    def retrieve(self, query: str, max_results: int = 5) -> List[Dict]:
        """
        Agentic retrieval: search -> evaluate -> reformulate if needed -> search again.
        Fully implemented in nb 33.
        """
        print(f"  [stub] Would retrieve top-{max_results} docs for: {query[:60]}...")
        return []

    def assess_relevance(self, doc: Dict, query: str) -> float:
        """Use LLM to judge document relevance. Implemented in nb 33."""
        print(f"  [stub] Would assess relevance of doc to query")
        return 0.0

    def process(self, state: SharedState) -> AgentMessage:
        """Retrieve documents for all search queries."""
        all_results = []
        for query in state.search_queries:
            results = self.retrieve(query)
            all_results.extend(results)

        state.retrieved_docs = all_results
        return self._create_message(
            to="orchestrator", msg_type="response",
            content={"retrieved_docs": len(all_results)}
        )

print("✓ RetrievalAgent skeleton defined")
retriever = RetrievalAgent()
print(f"  {retriever}")

In [ ]:
# ─── Skeleton: AnalysisAgent ───

class AnalysisAgent(AnalyzerInterface):
    """Synthesizes findings and detects contradictions."""

    def __init__(self):
        super().__init__(name="analyzer")

    def synthesize(self, docs: List[Dict], question: str) -> List[Dict]:
        """Extract key findings grouped by theme. Implemented in nb 34."""
        print(f"  [stub] Would synthesize findings from {len(docs)} docs")
        return []

    def detect_contradictions(self, docs: List[Dict]) -> List[Dict]:
        """Compare claims across sources. Implemented in nb 34."""
        print(f"  [stub] Would check {len(docs)} docs for contradictions")
        return []

    def weigh_evidence(self, docs: List[Dict]) -> Dict[str, float]:
        """Rate source quality. Implemented in nb 34."""
        print(f"  [stub] Would weigh evidence quality for {len(docs)} docs")
        return {}

    def identify_gaps(self, docs: List[Dict], question: str) -> List[str]:
        """Find uncovered aspects of the question. Implemented in nb 34."""
        print(f"  [stub] Would identify gaps in coverage")
        return []

    def process(self, state: SharedState) -> AgentMessage:
        """Run full analysis pipeline."""
        findings = self.synthesize(state.retrieved_docs, state.research_question)
        contradictions = self.detect_contradictions(state.retrieved_docs)
        gaps = self.identify_gaps(state.retrieved_docs, state.research_question)

        state.findings = findings
        state.contradictions = contradictions
        state.gaps = gaps

        return self._create_message(
            to="orchestrator", msg_type="response",
            content={
                "findings": len(findings),
                "contradictions": len(contradictions),
                "gaps": len(gaps)
            }
        )

print("✓ AnalysisAgent skeleton defined")
analyzer = AnalysisAgent()
print(f"  {analyzer}")

In [ ]:
# ─── Skeleton: WritingAgent ───

class WritingAgent(WriterInterface):
    """Produces structured research reports."""

    def __init__(self):
        super().__init__(name="writer")

    def plan_structure(self, analysis: Dict) -> List[Dict]:
        """Create report outline from analysis. Implemented in nb 35."""
        print(f"  [stub] Would plan report structure from analysis")
        return []

    def write_section(self, section_plan: Dict, findings: List[Dict]) -> str:
        """Write one section with citations. Implemented in nb 35."""
        print(f"  [stub] Would write section: {section_plan.get('title', 'untitled')}")
        return ""

    def write_introduction(self, question: str, overview: str) -> str:
        """Write the introduction. Implemented in nb 35."""
        print(f"  [stub] Would write introduction")
        return ""

    def write_conclusion(self, findings: List[Dict]) -> str:
        """Write the conclusion. Implemented in nb 35."""
        print(f"  [stub] Would write conclusion")
        return ""

    def assemble_report(self, sections: List[str], question: str) -> str:
        """Combine all sections into a final report. Implemented in nb 35."""
        print(f"  [stub] Would assemble {len(sections)} sections into report")
        return ""

    def process(self, state: SharedState) -> AgentMessage:
        """Run full writing pipeline."""
        analysis = {
            "findings": state.findings,
            "contradictions": state.contradictions,
            "gaps": state.gaps,
        }
        structure = self.plan_structure(analysis)
        sections = [self.write_section(s, state.findings) for s in structure]
        report = self.assemble_report(sections, state.research_question)
        state.draft_report = report
        return self._create_message(
            to="orchestrator", msg_type="response",
            content={"report_length": len(report), "sections": len(sections)}
        )

print("✓ WritingAgent skeleton defined")
writer = WritingAgent()
print(f"  {writer}")

In [ ]:
# ─── Skeleton: ReviewAgent ───

class ReviewAgent(ReviewerInterface):
    """Evaluates report quality and generates feedback."""

    RUBRIC = {
        "accuracy": "Are claims supported by citations? Are facts correct?",
        "completeness": "Does the report fully address the research question?",
        "clarity": "Is the writing clear, well-organized, and easy to follow?",
        "citation_quality": "Are sources properly attributed? Are citations relevant?",
        "coherence": "Do sections flow logically? Is there a clear narrative?"
    }

    def __init__(self):
        super().__init__(name="reviewer")

    def review(self, report: str, rubric: Dict = None) -> Dict:
        """Score the report on each rubric dimension. Implemented in nb 36."""
        rubric = rubric or self.RUBRIC
        print(f"  [stub] Would review report ({len(report)} chars) on {len(rubric)} dimensions")
        return {"scores": {}, "feedback": [], "needs_revision": False}

    def score_dimension(self, report: str, dimension: str, criteria: str) -> Tuple[int, str]:
        """Score one dimension 1-5 with feedback. Implemented in nb 36."""
        print(f"  [stub] Would score '{dimension}'")
        return (3, "Stub feedback")

    def needs_revision(self, scores: Dict, threshold: float = 3.5) -> bool:
        """Check if any dimension < 3 or average < threshold."""
        if not scores:
            return True
        avg = sum(scores.values()) / len(scores)
        any_low = any(v < 3 for v in scores.values())
        return any_low or avg < threshold

    def process(self, state: SharedState) -> AgentMessage:
        """Review the current draft report."""
        result = self.review(state.draft_report)
        state.review_scores = result.get("scores", {})
        state.review_feedback = result.get("feedback", [])
        state.needs_revision = result.get("needs_revision", False)
        return self._create_message(
            to="orchestrator", msg_type="response",
            content=result
        )

print("✓ ReviewAgent skeleton defined")
reviewer = ReviewAgent()
print(f"  {reviewer}")
print(f"  Rubric dimensions: {list(ReviewAgent.RUBRIC.keys())}")

## 9. Workflow Sequence — The Orchestration Logic

The Orchestrator coordinates all agents in a specific sequence with decision points.

In [ ]:
# ─── Orchestrator Skeleton ───

class ScholarOrchestrator:
    """
    Coordinates the Castalia Scholar pipeline:
    Plan -> Retrieve -> Analyze -> Write -> Review -> [Revise?]
    """

    def __init__(self):
        self.planner = PlannerAgent()
        self.retriever = RetrievalAgent()
        self.analyzer = AnalysisAgent()
        self.writer = WritingAgent()
        self.reviewer = ReviewAgent()
        self.state = None

    def run(self, research_question: str) -> SharedState:
        """Execute the full research pipeline."""
        self.state = SharedState(research_question=research_question)
        print(f"\n{'='*70}")
        print(f"  CASTALIA SCHOLAR — Research Pipeline")
        print(f"{'='*70}")
        print(f"  Question: {research_question}")
        print(f"{'='*70}\n")

        # Step 1: Plan
        print("Step 1: PLANNING — Decomposing research question...")
        msg = self.planner.process(self.state)
        self.state.log_message(msg)
        print(f"  ✓ Generated {len(self.state.sub_questions)} sub-questions")
        for i, sq in enumerate(self.state.sub_questions, 1):
            print(f"    {i}. {sq}")

        # Step 2: Retrieve
        print("\nStep 2: RETRIEVAL — Finding relevant documents...")
        msg = self.retriever.process(self.state)
        self.state.log_message(msg)
        print(f"  ✓ Retrieved {len(self.state.retrieved_docs)} documents")

        # Step 3: Analyze
        print("\nStep 3: ANALYSIS — Synthesizing findings...")
        msg = self.analyzer.process(self.state)
        self.state.log_message(msg)
        print(f"  ✓ Found {len(self.state.findings)} findings, "
              f"{len(self.state.contradictions)} contradictions, "
              f"{len(self.state.gaps)} gaps")

        # Step 4: Write
        print("\nStep 4: WRITING — Generating research report...")
        msg = self.writer.process(self.state)
        self.state.log_message(msg)
        print(f"  ✓ Draft report: {len(self.state.draft_report)} characters")

        # Step 5: Review + Revision Loop
        for revision in range(self.state.max_revisions + 1):
            cycle_label = "initial" if revision == 0 else f"revision {revision}"
            print(f"\nStep 5.{revision}: REVIEW ({cycle_label})...")
            msg = self.reviewer.process(self.state)
            self.state.log_message(msg)

            if not self.state.needs_revision:
                print(f"  ✓ Report ACCEPTED (scores: {self.state.review_scores})")
                break
            elif revision < self.state.max_revisions:
                print(f"  ✗ Report needs revision (scores: {self.state.review_scores})")
                print(f"    Feedback: {self.state.review_feedback[:2]}")
                print(f"\nStep 5.{revision}r: REVISING...")
                msg = self.writer.process(self.state)
                self.state.log_message(msg)
                self.state.revision_count += 1
            else:
                print(f"  ⚠ Max revisions reached. Using best draft.")

        # Finalize
        self.state.final_report = self.state.draft_report
        print(f"\n{'='*70}")
        print(f"  PIPELINE COMPLETE")
        print(f"  Revisions: {self.state.revision_count}")
        print(f"  Agent calls: {dict(self.state.agent_call_count)}")
        print(f"  Messages: {len(self.state.message_log)}")
        print(f"  Time: {self.state.elapsed_time():.1f}s")
        print(f"{'='*70}")

        return self.state


print("✓ ScholarOrchestrator defined")
print("  Pipeline: Plan -> Retrieve -> Analyze -> Write -> Review -> [Revise?]")
print("  Max revision cycles: 2")

### 9.1 — Workflow Decision Tree

```
START
  │
  ▼
[Planner] ──► sub-questions
  │
  ▼
[Retriever] ──► documents + citations
  │
  ▼
[Analyzer] ──► findings + contradictions + gaps
  │
  ├──► gaps found? ──YES──► [Retriever] (additional search)
  │                          │
  │                          ▼
  │                   [Analyzer] (re-analyze with new docs)
  │
  ▼
[Writer] ──► draft report
  │
  ▼
[Reviewer] ──► scores + feedback
  │
  ├──► needs revision? ──YES──► revision < max?
  │                              │
  │                     YES◄─────┘───────►NO
  │                      │                 │
  │                      ▼                 ▼
  │               [Writer] (revise)   Use best draft
  │                      │
  │                      ▼
  │               [Reviewer] (re-evaluate)
  │                      │
  │                      └──► (loop back to decision)
  │
  ▼ (accepted)
FINAL REPORT ──► User
```

## 10. Dry Run — Verify the Pipeline Structure

Let's run the orchestrator with stub implementations to verify the workflow logic is correct.

In [ ]:
# ─── Dry Run with Stubs ───
# The planner will use the LLM to actually decompose the question.
# All other agents use stub methods (they'll be implemented in later notebooks).

orchestrator = ScholarOrchestrator()
result_state = orchestrator.run(
    "What are the main approaches to making AI systems more energy-efficient?"
)

print("\n--- State Summary After Dry Run ---")
print(result_state.summary())

## 11. Testing Strategy

Each agent will be tested independently in its own notebook. Here's the testing plan:

### Per-Agent Tests

| Agent | Test Approach | Notebook |
|-------|--------------|----------|
| **Planner** | Give 3 questions → verify sub-questions are relevant and non-overlapping | 32 (below) |
| **Retriever** | Build corpus → query → verify top-k relevance > threshold | 33 |
| **Analyzer** | Feed docs with known contradictions → verify detection | 34 |
| **Writer** | Give pre-analyzed findings → verify report has citations, structure | 35 |
| **Reviewer** | Give report with planted errors → verify specific feedback | 36 |

### Integration Tests (Notebook 37)

1. **Happy path** — well-formed question → complete pipeline → quality report
2. **Revision path** — planted low-quality draft → reviewer catches it → revision improves it
3. **Comparison** — same question through Castalia Scholar vs single ReAct agent vs simple RAG

In [ ]:
# ─── Test the Planner (the only agent using LLM in this notebook) ───

test_questions = [
    "What are the main approaches to making AI systems more energy-efficient?",
    "How has natural language processing evolved from rule-based to neural approaches?",
    "What are the ethical implications of using AI in criminal justice?",
]

print("Testing PlannerAgent on 3 research questions:\n")
planner_test = PlannerAgent()

for i, question in enumerate(test_questions, 1):
    print(f"{'='*60}")
    print(f"Test {i}: {question}")
    print(f"{'='*60}")
    result = planner_test.decompose(question)
    sub_qs = result.get("sub_questions", [])
    queries = result.get("search_queries", [])
    print(f"  Sub-questions ({len(sub_qs)}):")
    for j, sq in enumerate(sub_qs, 1):
        print(f"    {j}. {sq}")
    print(f"  Search queries ({len(queries)}):")
    for j, q in enumerate(queries, 1):
        print(f"    {j}. {q}")
    print()

## 12. Key Deliverables — What We Built

This notebook defines the complete architecture of Castalia Scholar:

### Artifacts Created

| Artifact | Purpose |
|----------|---------|
| `AgentMessage` | Standardized inter-agent communication format |
| `SharedState` | Blackboard pattern — shared state for all agents |
| `BaseAgent` + 5 interfaces | Abstract base classes defining contracts |
| 7 data classes | Structured types (RetrievedDoc, Citation, Finding, etc.) |
| 5 skeleton agents | Concrete stubs ready for implementation |
| `ScholarOrchestrator` | Pipeline coordinator with revision loops |

### What's Next

| Next Notebook | What Gets Implemented |
|---------------|----------------------|
| **33** — Retrieval Agent | Build document corpus, FAISS index, agentic RAG with citations |
| **34** — Analysis Agent | Synthesis, contradiction detection, gap identification |
| **35** — Writing Agent | Report structure planning, section writing, citation formatting |
| **36** — Review Agent | Multi-dimensional rubric scoring, actionable feedback |
| **37** — Full System | Wire everything together, end-to-end runs, evaluation |

## Key Takeaways

1. **Architecture first** — define interfaces before implementations. This prevents the "big ball of mud" problem.
2. **The blackboard pattern** — agents read/write shared state rather than passing messages directly. Simpler to debug.
3. **Standardized messages** — every agent speaks the same language (AgentMessage format).
4. **Abstract base classes** — enforce contracts so implementations are interchangeable.
5. **Bounded revision loops** — the reviewer can request revisions, but max 2 cycles prevents infinite loops.
6. **Data classes** — structured types (RetrievedDoc, Citation, Finding) make inter-agent data flows explicit.

> **Next notebook**: We build the **Retrieval Agent** — agentic RAG with citation tracking and iterative search.